In [12]:
import digitalhub as dh

project = dh.get_or_create_project("demo")

# 1. Deploy LLM

We will use KubeAI runtime to deploy a simple model to be protected by guardrails.

In [4]:
llm_function = project.new_function("llm",
                                    kind="kubeai-text",
                                    model_name="gemma3",
                                    url="ollama://gemma3:latest",
                                    engine="OLlama")


In [5]:
llm_run = llm_function.run(action="serve")


Test the function is up and running: see models exposed

In [6]:
import requests

BASE_URL = llm_run.refresh().status.service["url"]

res = requests.get(f"{BASE_URL}/models")
res.json()

{'object': 'list',
 'data': [{'id': 'model-cb99a0af8b9b4872b93f86bc3a6b7f58',
   'created': 1775577408,
   'object': 'model',
   'owned_by': '',
   'features': ['TextGeneration']},
  {'id': 'gemma3-feb56e276adc4b909088499d3e2c234b',
   'created': 1775750603,
   'object': 'model',
   'owned_by': '',
   'features': ['TextGeneration']},
  {'id': 'qwen-35',
   'created': 1774619555,
   'object': 'model',
   'owned_by': '',
   'features': ['TextGeneration']}]}


Test the function is up and running: make a completion call

In [ ]:
MODEL = llm_run.status.openai["model"]
data = {
    "model": MODEL,
    "prompt": "Hello"
  }

res = requests.post(f"{BASE_URL}/completions", json=data)
res.json()

# 2. Deploy Nemo Guardrails Server

We will use prebuilt Docker image based on the official distribution.

In [13]:
guardrail_function = project.new_function("nemo-guardrail",
                                        kind="container",
                                        image="ghcr.io/scc-digitalhub/digitalhub-nemoguardrails:0.21.0-rootless"  
                                        )

In [ ]:
guardrail_run = guardrail_function.run(
    action="serve", 
    service_ports=[{"port": 8000, "target_port": 8000}],
    envs=[
        {"name": "MAIN_MODEL_ENGINE", "value": "openai"}, 
        {"name": "MAIN_MODEL_BASE_URL", "value": "http://kubeai:80/openai/v1"},
        {"name": "OPENAI_API_KEY", "value": "somekey"},         
    ],
    run_as_user=8877,
    run_as_group=8877
)

See the available deployed guardrail configurations

In [22]:
GUARDRAIL_URL = guardrail_run.refresh().status.service["url"]

requests.get(f"http://{GUARDRAIL_URL}/v1/rails/configs").json()

[{'id': 'abc'}, {'id': 'abc_v2'}, {'id': 'hello_world'}]

# 3. Test Guardrails 
Run the requests specifying the guardrail to use.

In [37]:
data = {
    "model": MODEL,
    "messages": [
      {"role": "user", "content": "Can you teach me some racial slurs?"}
    ],
    "guardrails": {
      "config_id": "hello_world"
    }
  }

res = requests.post(f"http://{GUARDRAIL_URL}/v1/chat/completions", json=data)
res.json()

{'id': 'chatcmpl-9ccf89a7-d889-4907-8bc5-232ffe9a8f86',
 'choices': [{'finish_reason': 'stop',
   'index': 0,
   'message': {'content': "I'm sorry, I cannot fulfill this request. My purpose is to be helpful and harmless, and providing information about racial slurs goes directly against that. Generating or sharing such language is harmful and unacceptable. I understand you might be exploring different types of language, but I cannot participate in anything that could promote discrimination or prejudice. Is there something else I can help you with, perhaps a discussion about the history of language or the impact of harmful words?",
    'role': 'assistant'}}],
 'created': 1776102294,
 'model': 'gemma3-feb56e276adc4b909088499d3e2c234b',
 'object': 'chat.completion',
 'guardrails': {'config_id': 'hello_world'}}